# Week 3: Gymnasium 环境 — 交互式教程
# Week 3: Gymnasium Environments — Interactive Tutorial

本教程将带你从零开始理解 Gymnasium 的核心概念，并通过动手实践掌握自定义环境的创建。

**学习目标：**
- 理解 Gymnasium 的 Spaces 系统
- 创建自定义 Gymnasium 环境
- 用 Q-Learning 在自定义环境中训练智能体
- 理解 `terminated` vs `truncated` 的区别
- 了解 Stable-Baselines3 的集成方式

## 环境准备 (Setup)

In [1]:
import os
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

# 输出目录
OUTPUT_DIR = os.getcwd() + "/courses/rl/notes/week3_gymnasium_complete_demo_pages"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ 环境准备完成")

✅ 环境准备完成


## 1. Gymnasium Spaces 系统

Gymnasium 用 `spaces` 模块描述观测空间和动作空间的**元信息**（类型、范围、维度）。

MDP 理论中，状态空间 $S$ 是抽象集合。但代码中，算法需要知道状态的类型、范围和维度。

| Space | 数学对应 | 用途 |
|-------|---------|------|
| `Discrete(n)` | $\{0, 1, ..., n-1\}$ | 离散动作/状态 |
| `Box(low, high, shape)` | $\mathbb{R}^d$ 的子集 | 连续空间 |
| `Dict({...})` | $S_1 \times S_2$ | 组合多个子空间 |

In [2]:
# Discrete Space: 离散空间 {0, 1, ..., n-1}
discrete_space = spaces.Discrete(4)
print(f"[Discrete(4)]")
print(f"  n = {discrete_space.n}")
print(f"  sample = {discrete_space.sample()}")
print(f"  contains(3) = {discrete_space.contains(3)}")
print(f"  contains(5) = {discrete_space.contains(5)}")

# Box Space: 连续空间
box_space = spaces.Box(low=0, high=10, shape=(2,), dtype=np.float32)
print(f"\n[Box(0, 10, shape=(2,))]")
print(f"  shape = {box_space.shape}")
print(f"  sample = {box_space.sample()}")

# Dict Space: 字典空间
dict_space = spaces.Dict({
    "agent": spaces.Discrete(12),
    "target": spaces.Discrete(12),
})
print(f"\n[Dict(agent=Discrete(12), target=Discrete(12))]")
print(f"  sample = {dict_space.sample()}")

[Discrete(4)]
  n = 4
  sample = 3
  contains(3) = True
  contains(5) = False

[Box(0, 10, shape=(2,))]
  shape = (2,)
  sample = [9.597388 8.856842]

[Dict(agent=Discrete(12), target=Discrete(12))]
  sample = {'agent': np.int64(10), 'target': np.int64(3)}


### 🧪 试一试 (Try It)

修改上面的代码：
1. 创建一个 `Discrete(10)` 空间，采样 5 次看看结果
2. 创建一个 `Box(-1, 1, shape=(3,))` 空间，观察采样值的范围
3. 用 `contains()` 检查一个值是否在空间内

## 2. 自定义 Gymnasium 环境

Gymnasium 环境必须继承 `gymnasium.Env` 并实现 5 个核心方法：

```
__init__()  → 定义 spaces，初始化
reset()     → 重置到初始状态，返回 (obs, info)
step()      → 执行动作，返回 (obs, reward, terminated, truncated, info)
render()    → 可视化
close()     → 释放资源
```

**关键点：**
- `reset()` 中必须调用 `super().reset(seed=seed)` 设置 `self.np_random`
- `step()` 返回 **5 个值**（Gymnasium 新增 `truncated`）
- 所有随机操作用 `self.np_random` 而不是 `np.random`

In [3]:
class SimpleGridWorldEnv(gym.Env):
    """
    4x3 GridWorld 环境
    ┌───┬───┬───┬───┐
    │ 0 │ 1 │ 2 │+1 │  目标在 (0,3)
    ├───┼───┼───┼───┤
    │ 4 │ W │ 6 │-1 │  墙在 (1,1), 悬崖在 (1,3)
    ├───┼───┼───┼───┤
    │ 8 │ 9 │10 │11 │  起点在 (2,0)=state 8
    └───┴───┴───┴───┘
    动作: 0=右, 1=上, 2=左, 3=下
    """
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, render_mode=None):
        super().__init__()
        self.rows, self.cols = 3, 4
        self.observation_space = spaces.Discrete(self.rows * self.cols)
        self.action_space = spaces.Discrete(4)
        self.render_mode = render_mode
        self.start, self.goal, self.cliff, self.wall = 8, 3, 7, 5
        self._action_to_delta = {0: (0,1), 1: (-1,0), 2: (0,-1), 3: (1,0)}

    def _state_to_rc(self, s):
        return s // self.cols, s % self.cols

    def _rc_to_state(self, r, c):
        return r * self.cols + c

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)  # 设置 self.np_random
        self._agent_pos = self.start
        return self._agent_pos, {}

    def step(self, action):
        r, c = self._state_to_rc(self._agent_pos)
        dr, dc = self._action_to_delta[action]
        nr, nc = max(0, min(self.rows-1, r+dr)), max(0, min(self.cols-1, c+dc))
        ns = self._rc_to_state(nr, nc)
        if ns == self.wall:
            ns = self._agent_pos
        self._agent_pos = ns
        if ns == self.goal:
            return ns, 1.0, True, False, {}
        elif ns == self.cliff:
            return ns, -1.0, True, False, {}
        return ns, -0.01, False, False, {}

    def render(self):
        if self.render_mode == "ansi":
            symbols = {self.goal: "+1", self.cliff: "-1", self.wall: " W"}
            grid = []
            for r in range(self.rows):
                row = []
                for c in range(self.cols):
                    s = self._rc_to_state(r, c)
                    if s == self._agent_pos:
                        row.append(" A ")
                    else:
                        row.append(f" {symbols.get(s, '.')} ")
                grid.append("|".join(row))
            return "\n".join(grid)

# 测试环境
env = SimpleGridWorldEnv(render_mode="ansi")
obs, info = env.reset(seed=42)
print(f"观测空间: {env.observation_space}")
print(f"动作空间: {env.action_space}")
print(f"初始状态: {obs}")
print(f"\n{env.render()}")

观测空间: Discrete(12)
动作空间: Discrete(4)
初始状态: 8

 . | . | . | +1 
 . |  W | . | -1 
 A | . | . | . 


## 3. Agent-Environment 交互循环

Sutton §3.1 将 RL 形式化为：每个时间步 $t$，Agent 选择 $A_t$，Environment 返回 $S_{t+1}, R_{t+1}$。

$$A_t \rightarrow \text{Environment} \rightarrow (S_{t+1}, R_{t+1})$$

Gymnasium 的 `step()` 就是这个循环的代码实现。

In [4]:
# 手动执行一个 episode
env = SimpleGridWorldEnv(render_mode="ansi")
obs, _ = env.reset(seed=42)
action_names = ["Right", "Up", "Left", "Down"]
action_symbols = ['→', '↑', '←', '↓']

# 最优路径: 上上右右右
actions = [1, 1, 0, 0, 0]
print("执行最优路径: 上→上→右→右→右")
print(f"初始: state={obs}")
print()

for a in actions:
    obs, reward, terminated, truncated, info = env.step(a)
    print(f"  {action_names[a]:5s} ({action_symbols[a]}) → state={obs:2d}, "
          f"reward={reward:+.2f}, terminated={terminated}")
    if terminated:
        print(f"\n🎯 到达目标! Episode 结束")
        break

执行最优路径: 上→上→右→右→右
初始: state=8

  Up    (↑) → state= 4, reward=-0.01, terminated=False
  Up    (↑) → state= 0, reward=-0.01, terminated=False
  Right (→) → state= 1, reward=-0.01, terminated=False
  Right (→) → state= 2, reward=-0.01, terminated=False
  Right (→) → state= 3, reward=+1.00, terminated=True

🎯 到达目标! Episode 结束


### 🧪 试一试 (Try It)

1. 修改动作序列，尝试走到悬崖 (state 7)，观察 `terminated` 和 `reward`
2. 尝试撞墙（从 state 4 向右走），观察状态是否改变
3. 设计一条从 state 8 到 goal 的不同路径

## 4. Q-Learning 训练

在自定义 Gymnasium 环境中训练 Q-Learning 智能体。

Q-Learning 更新规则：
$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

In [5]:
# Q-Learning 训练
alpha, gamma = 0.1, 0.99
epsilon, epsilon_min, epsilon_decay = 1.0, 0.01, 0.995
num_episodes = 500

env = SimpleGridWorldEnv()
qtable = np.zeros((env.observation_space.n, env.action_space.n))
rewards_history, steps_history = [], []

for ep in range(num_episodes):
    obs, _ = env.reset(seed=ep)
    total_reward, steps = 0, 0
    for _ in range(100):
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(qtable[obs]))
        next_obs, reward, terminated, truncated, _ = env.step(action)
        best_next = np.max(qtable[next_obs]) if not terminated else 0
        qtable[obs, action] += alpha * (reward + gamma * best_next - qtable[obs, action])
        obs = next_obs
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    rewards_history.append(total_reward)
    steps_history.append(steps)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print(f"训练完成: {num_episodes} episodes")
print(f"最后 50 episodes 平均奖励: {np.mean(rewards_history[-50:]):.3f}")
print(f"最后 50 episodes 平均步数: {np.mean(steps_history[-50:]):.1f}")

训练完成: 500 episodes
最后 50 episodes 平均奖励: 0.956
最后 50 episodes 平均步数: 5.4


In [6]:
# 可视化训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
window = 20
rewards_smooth = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(rewards_smooth, color='#3498db', linewidth=1.5)
ax1.set_title("Training Reward", fontsize=12)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Total Reward (smoothed)")
ax1.axhline(y=0.9, color='#2ecc71', linestyle='--', alpha=0.5, label='Target')
ax1.legend()

steps_smooth = np.convolve(steps_history, np.ones(window)/window, mode='valid')
ax2.plot(steps_smooth, color='#e74c3c', linewidth=1.5)
ax2.set_title("Steps per Episode", fontsize=12)
ax2.set_xlabel("Episode")
ax2.set_ylabel("Steps (smoothed)")
plt.tight_layout()
plt.show()

C:\Users\40270\AppData\Local\Temp\ipykernel_73804\1859580044.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Q-Table 可视化与最优策略

训练完成后，Q-Table 中存储了每个 (state, action) 对的价值。
$V(s) = \max_a Q(s,a)$ 是状态价值函数。

In [7]:
# 打印 Q-Table
action_symbols = ['→', '↑', '←', '↓']
print("Learned Q-Table:")
print(f"{'State':>6} | {'Right':>8} {'Up':>8} {'Left':>8} {'Down':>8} | Best")
print("-" * 65)
for s in range(12):
    if s == 5:
        print(f"  {s:>3}  |   WALL                                  |  W")
        continue
    q = qtable[s]
    best = int(np.argmax(q))
    print(f"  {s:>3}  | {q[0]:>8.3f} {q[1]:>8.3f} {q[2]:>8.3f} {q[3]:>8.3f} | {action_symbols[best]}")

Learned Q-Table:
 State |    Right       Up     Left     Down | Best
-----------------------------------------------------------------
    0  |    0.960    0.940    0.940    0.913 | →
    1  |    0.980    0.954    0.938    0.959 | →
    2  |    1.000    0.979    0.959    0.948 | →
    3  |    0.000    0.000    0.000    0.000 | →
    4  |    0.920    0.941    0.920    0.900 | ↑
    5  |   WALL                                  |  W
    6  |   -0.902    0.979    0.568    0.340 | ↑
    7  |    0.000    0.000    0.000    0.000 | →
    8  |    0.870    0.921    0.896    0.896 | ↑
    9  |    0.478    0.458    0.899    0.597 | ←
   10  |    0.044    0.788    0.211    0.265 | ↑
   11  |   -0.007   -0.686    0.153    0.010 | ←


In [8]:
# 可视化 Q-Table 和策略
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# V(s) 热力图
q_max = np.max(qtable, axis=1).reshape(3, 4)
q_max[1, 1] = np.nan
im = ax1.imshow(q_max, cmap='RdYlGn', aspect='equal')
ax1.set_title("V(s) = max_a Q(s,a)", fontsize=12)
for r in range(3):
    for c in range(4):
        s = r * 4 + c
        if s == 5:
            ax1.text(c, r, "WALL", ha='center', va='center', fontsize=10, fontweight='bold')
        elif s == 3:
            ax1.text(c, r, f"GOAL\n{q_max[r,c]:.2f}", ha='center', va='center', fontsize=9)
        elif s == 7:
            ax1.text(c, r, f"CLIFF\n{q_max[r,c]:.2f}", ha='center', va='center', fontsize=9)
        else:
            ax1.text(c, r, f"s={s}\n{q_max[r,c]:.2f}", ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax1, shrink=0.8)

# 策略箭头
arrow_dx = {0: 0.3, 1: 0, 2: -0.3, 3: 0}
arrow_dy = {0: 0, 1: -0.3, 2: 0, 3: 0.3}
ax2.set_xlim(-0.5, 3.5); ax2.set_ylim(-0.5, 2.5)
ax2.set_aspect('equal'); ax2.invert_yaxis()
ax2.set_title("Optimal Policy", fontsize=12)
for r in range(3):
    for c in range(4):
        s = r * 4 + c
        if s == 5:
            ax2.add_patch(plt.Rectangle((c-0.4, r-0.4), 0.8, 0.8, color='gray', alpha=0.5))
        elif s == 3:
            ax2.add_patch(plt.Rectangle((c-0.4, r-0.4), 0.8, 0.8, color='#2ecc71', alpha=0.3))
            ax2.text(c, r, "+1", ha='center', va='center', fontsize=14, fontweight='bold', color='green')
        elif s == 7:
            ax2.add_patch(plt.Rectangle((c-0.4, r-0.4), 0.8, 0.8, color='#e74c3c', alpha=0.3))
            ax2.text(c, r, "-1", ha='center', va='center', fontsize=14, fontweight='bold', color='red')
        else:
            best = int(np.argmax(qtable[s]))
            ax2.arrow(c, r, arrow_dx[best], arrow_dy[best],
                      head_width=0.12, head_length=0.06, fc='#3498db', ec='#2c3e50', lw=1.5)
for i in range(5): ax2.axvline(x=i-0.5, color='gray', lw=0.5, alpha=0.5)
for i in range(4): ax2.axhline(y=i-0.5, color='gray', lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

C:\Users\40270\AppData\Local\Temp\ipykernel_73804\1570943423.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 🧪 试一试 (Try It)

1. 修改 `alpha`（学习率）为 0.5，重新训练，观察收敛速度变化
2. 修改 `gamma`（折扣因子）为 0.5，观察策略是否改变
3. 将悬崖奖励从 -1 改为 -10，观察智能体是否更加"谨慎"（绕远路避开悬崖附近）

## 6. terminated vs truncated

这是 Gymnasium 相对于旧版 Gym 的重要改进：

| | terminated | truncated |
|---|---|---|
| 含义 | 任务自然结束 | 人为截断 |
| 触发 | 到达目标/掉入悬崖 | 超过 max_episode_steps |
| MDP 意义 | 终止状态 | 非 MDP 概念 |
| Q-Learning | Q(terminal) = 0 | Q(s) ≠ 0 |

**为什么这很重要？** 如果把 `truncated` 当作 `terminated` 处理，Q-Learning 会错误地将截断状态的价值设为 0，导致学习不稳定。

In [9]:
# 演示 terminated
env = SimpleGridWorldEnv()
obs, _ = env.reset(seed=0)
print("场景 1: terminated (到达目标)")
for a in [1, 1, 0, 0, 0]:  # 上上右右右
    obs, reward, terminated, truncated, _ = env.step(a)
    if terminated:
        print(f"  State {obs}: terminated={terminated}, truncated={truncated}, reward={reward:+.2f}")
        print(f"  任务自然结束!")
        break

print()
print("场景 2: truncated (概念演示)")
print("  当 register() 设置 max_episode_steps=50 时,")
print("  超过 50 步未完成 → Gymnasium 自动设置 truncated=True")
print("  注意: truncated 由 Gymnasium wrapper 处理, 不在 env.step() 中实现")

print()
print("正确的 Q-Learning 处理方式:")
print("  if terminated:")
print("      target = reward  # 终止状态, 不 bootstrap")
print("  else:  # 包括 truncated")
print("      target = reward + gamma * max(Q[next_state])  # 继续 bootstrap")

场景 1: terminated (到达目标)
  State 3: terminated=True, truncated=False, reward=+1.00
  任务自然结束!

场景 2: truncated (概念演示)
  当 register() 设置 max_episode_steps=50 时,
  超过 50 步未完成 → Gymnasium 自动设置 truncated=True
  注意: truncated 由 Gymnasium wrapper 处理, 不在 env.step() 中实现

正确的 Q-Learning 处理方式:
  if terminated:
      target = reward  # 终止状态, 不 bootstrap
  else:  # 包括 truncated
      target = reward + gamma * max(Q[next_state])  # 继续 bootstrap


## 7. 观测空间设计：Dict vs Discrete

Slides 展示了三种观测空间设计，核心权衡是**可读性 vs 兼容性**：

| 方式 | SB3 Policy | Q-Table 兼容 |
|------|-----------|-------------|
| Dict + Box | MultiInputPolicy | ❌ |
| Dict + Discrete | MultiInputPolicy | ❌ |
| Single Discrete | MlpPolicy | ✅ |

⚠️ **关键陷阱：** Policy 必须匹配观测空间类型！

In [10]:
# Dict vs Discrete 编码对比
print("Dict 观测空间:")
print("  spaces.Dict({'agent': Discrete(12), 'target': Discrete(12)})")
print("  → SB3 需要 MultiInputPolicy")
print("  → Q-Table 不能直接索引")

print()
print("Discrete 观测空间:")
print("  spaces.Discrete(144)  # 12 * 12")
print("  → 编码: state = agent_pos * 12 + target_pos")
print("  → SB3 用 MlpPolicy")
print("  → Q-Table 直接索引: qtable[state, action]")

print()
# 编码示例
agent_pos, target_pos = 8, 3
encoded = agent_pos * 12 + target_pos
decoded_agent = encoded // 12
decoded_target = encoded % 12
print(f"编码: agent={agent_pos}, target={target_pos} → state={encoded}")
print(f"解码: state={encoded} → agent={decoded_agent}, target={decoded_target}")

Dict 观测空间:
  spaces.Dict({'agent': Discrete(12), 'target': Discrete(12)})
  → SB3 需要 MultiInputPolicy
  → Q-Table 不能直接索引

Discrete 观测空间:
  spaces.Discrete(144)  # 12 * 12
  → 编码: state = agent_pos * 12 + target_pos
  → SB3 用 MlpPolicy
  → Q-Table 直接索引: qtable[state, action]

编码: agent=8, target=3 → state=99
解码: state=99 → agent=8, target=3


## 8. Stable-Baselines3 集成概览

有了标准 Gymnasium 接口，就可以直接使用工业级算法：

```python
from stable_baselines3 import DQN, PPO, A2C

# 所有算法共享相同 API
model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)
model.save("model_name")
model = DQN.load("model_name")
action, _ = model.predict(obs)
```

| 算法 | 类型 | 动作空间 |
|------|------|---------|
| DQN | Off-policy, Value-based | 仅 Discrete |
| PPO | On-policy, Policy Gradient | Discrete + Continuous |
| A2C | On-policy, Actor-Critic | Discrete + Continuous |

> 注意：SB3 需要单独安装 (`pip install stable-baselines3`)，本教程不运行 SB3 代码。

## 9. 知识检查 (Knowledge Check)

回答以下问题，检验你对 Gymnasium 的理解：

**Q1:** `env.step(action)` 返回几个值？分别是什么？

<details>
<summary>答案</summary>
5 个值：observation, reward, terminated, truncated, info
</details>

**Q2:** `terminated=True` 和 `truncated=True` 有什么区别？

<details>
<summary>答案</summary>
terminated = 任务自然结束（到达目标/掉入悬崖），truncated = 人为截断（超过最大步数）
</details>

**Q3:** 如果观测空间是 `spaces.Dict({...})`，SB3 应该用什么 Policy？

<details>
<summary>答案</summary>
MultiInputPolicy（不是 MlpPolicy）
</details>

**Q4:** `reset()` 中为什么要调用 `super().reset(seed=seed)`？

<details>
<summary>答案</summary>
设置 self.np_random（NumPy RandomGenerator），确保环境内的随机操作可通过 seed 复现
</details>

**Q5:** BlocksWorld 有 30 个状态和 6 个动作，Q-Table 有多少个元素？

<details>
<summary>答案</summary>
30 × 6 = 180 个 Q 值
</details>

---

## 📚 参考资料

- [Gymnasium 官方文档](https://gymnasium.farama.org/)
- [Gymnasium 环境创建教程](https://gymnasium.farama.org/tutorials/gymnasium_basics/environment_creation/)
- [Stable-Baselines3 文档](https://stable-baselines3.readthedocs.io/)
- Sutton & Barto §3.1 — The Agent-Environment Interface
- Week 3 Slides: `week3_gymnasium_slides.md`
- Week 3 Storyline: `week3_gymnasium_storyline.md`
- Week 3 Tutorial: `week3_gymnasium_tutorial.md`